In [ ]:
"""
檔名：Q7yahoo電子商城耳機.py
功能：網頁資料擷取練習
學習重點：requests、BeautifulSoup、urllib3 網頁爬取與解析
"""
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

In [ ]:
options = Options()
options.binary_location = "/Applications/Brave Browser.app/Contents/MacOS/Brave Browser"

In [ ]:
driver = webdriver.Chrome(options=options)

In [ ]:
try:
    driver.get("https://tw.buy.yahoo.com/search/product?p=耳機")

    # 等到商品載入
    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[class*="sc-"]')))
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # 找所有含 $ 的 span，前面一個 span 就是商品名稱
    results = []

    # 找商品名稱：包含索尼/耳機等關鍵字的長文字 span
    # 找價格：緊接在 $ 符號後面的數字
    price_signs = soup.find_all("span", string="$")

    for sign in price_signs:
        try:
            # 價格數字在 $ 的父層裡
            parent = sign.parent
            full_text = parent.get_text().replace(",", "").strip()
            price_num = "".join(filter(str.isdigit, full_text))
            if not price_num:
                continue
            price = int(price_num)

            # 商品名稱往上找
            card = parent
            for _ in range(6):
                card = card.parent
                if card is None:
                    break
                name_span = card.find(
                    "span", attrs={"class": lambda c: c and len(c) > 10}
                )
                if name_span and len(name_span.get_text()) > 5:
                    name = name_span.get_text().strip()
                    break
            else:
                name = "未知商品"

            if price <= 1000:
                results.append((price, name))
        except:
            continue

    # 去重後排序
    seen = set()
    unique = []
    for price, name in results:
        if name not in seen:
            seen.add(name)
            unique.append((price, name))

    unique.sort(key=lambda x: x[0])

    print(f"價格 ≤ 1000 的耳機商品（由小到大）：\n")
    if unique:
        for price, name in unique:
            print(f"${price:>5}  {name}")
    else:
        print("無符合條件的商品")

In [ ]:
finally:
    driver.quit()